# Coherence Under Change: Llama-3.1-8B baseline (C0 + C3) on Kaggle

Runs Meta's Llama-3.1-8B-Instruct (4-bit) on the **same 3 abilene sequences** (seeds 100-102, 15 steps) that
Qwen2.5-Coder-7B already ran, under **C0** (no guard) and **C3** (full guard). Same judge, same prompts, so the
results line up with Qwen's step by step. Expected GPU time: about 2-3 hours.

## One-time setup (about 15 minutes)
1. **Kaggle account, phone verified:** kaggle.com → profile picture → Settings → Phone Verification. Without it the
   GPU and Internet switches stay grey.
2. **Llama access on Hugging Face:** log in at huggingface.co, open
   https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct and accept the licence. Wait until the page says
   you have been granted access (usually minutes, sometimes a few hours).
3. **Hugging Face token:** huggingface.co → Settings → Access Tokens → Create new token → type **Read** → copy it.
   Never paste it into a notebook cell or a chat.
4. **Project dataset:** kaggle.com → Create → New Dataset → upload `coherence-under-change.zip` → name it
   `coherence-under-change` → Create.
5. **Import this notebook:** kaggle.com → Create → New Notebook → File → Import Notebook → upload
   `kaggle_llama.ipynb`.
6. **In the notebook:**
   - right panel → Session options → Accelerator **GPU T4 x2**, Internet **On**;
   - right panel → Add Input → Your Datasets → `coherence-under-change`;
   - top menu Add-ons → Secrets → Add a new secret: label `HF_TOKEN`, value = your token → tick it so it is
     attached to this notebook.

## Running it
- **Quick test first (interactive, about 10 minutes):** run cells 1 to 5 one by one. Cell 4 must print
  `Llama access OK`, cell 5 must end with `OK` and `all requested runs are complete`.
- **The real run (background):** click **Save Version → Save & Run All (Commit) → Save**, then close the tab.
  Do *not* run cell 6 in the interactive editor: Kaggle stops idle interactive sessions after about 40 minutes and
  everything in it is lost. A saved version keeps running for up to 12 hours with the tab closed.
- **Watch progress:** open the version (number next to Save Version) → **Logs**. Each finished step prints a line;
  a step can take 1-20 minutes.
- **When it says complete:** version → **Output** → download `cuc_runs.zip` and send it to Akshith.

## If it stops before finishing
Every finished step is saved, and a new run resumes from the last one. In the editor: Add Input → Your Work →
Notebooks → this notebook (its last version's output is attached), then Save & Run All again. Cell 2 picks the
earlier results up automatically.


In [ ]:
# 1. GPU check: expect two lines 'Tesla T4, 15360 MiB'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Project + earlier results (read-only inputs are copied to the writable /kaggle/working)
import glob, os, shutil
src = [p for p in glob.glob('/kaggle/input/**/cuc', recursive=True)
       if os.path.isdir(p) and os.path.exists(os.path.join(os.path.dirname(p), 'scripts'))]
assert src, 'Add the coherence-under-change dataset as an input (right panel -> Add Input)'
PROJECT = os.path.dirname(src[0])
shutil.rmtree('/kaggle/working/cuc', ignore_errors=True)
shutil.copytree(PROJECT, '/kaggle/working/cuc', ignore=shutil.ignore_patterns('runs', 'results', 'zoo', '__pycache__', '*.png'))
OUT = '/kaggle/working/cuc_runs'
prev = [p for p in glob.glob('/kaggle/input/**/cuc_runs', recursive=True) if os.path.isdir(os.path.join(p, 'runs'))]
if prev and not os.path.exists(OUT):
    shutil.copytree(prev[0], OUT)
    print('continuing from earlier runs in', prev[0])
os.makedirs(OUT, exist_ok=True)
MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
NAME = 'llama3.1-8b'                      # results go to cuc_runs/runs/llama3.1-8b/
%cd /kaggle/working/cuc
print('project:', PROJECT)

In [ ]:
# 3. Libraries (about 2 minutes; red dependency-conflict warnings about bigframes/gradio are harmless)
!pip -q install -U transformers accelerate bitsandbytes peft networkx

In [ ]:
# 4. Hugging Face login from the Kaggle secret HF_TOKEN, and a check that this account may download Llama
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download
try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    raise RuntimeError('No secret HF_TOKEN attached: top menu Add-ons -> Secrets -> add HF_TOKEN and tick it') from e
login(token)                       # saved for the run in cell 6 as well
os.environ['HF_TOKEN'] = token
try:
    hf_hub_download(MODEL, 'config.json')
    print('Llama access OK')
except Exception as e:
    raise RuntimeError('Token works but this account has no access to ' + MODEL +
                     ' yet: accept the licence on its Hugging Face page and wait for approval. (' + type(e).__name__ + ')')

In [ ]:
# 5. Pipeline check without a model (about 1 minute): tests + a perfect stand-in model must succeed on every step
!python3 -m unittest discover -s tests 2>&1 | tail -3
!python3 scripts/run_models.py --backend gold --bed abilene --seeds 100-100 --length 10 --conds C0,C3 --out /kaggle/working/check

In [ ]:
# 6. The Llama runs: C0 and C3 on the same 3 abilene sequences as Qwen (finished steps are skipped on a rerun).
#    About 2-3 hours.  Run this through Save Version -> Save & Run All, not in the interactive editor.
!python3 scripts/run_models.py --backend hf --model {MODEL} --name {NAME} --bed abilene --seeds 100-102 --length 15 --conds C0,C3 --workers 3 --out {OUT}

In [ ]:
# 7. Results table + one zip to download (version -> Output -> cuc_runs.zip)
!python3 scripts/summarize_runs.py {OUT} --out {OUT}/summary
!cd /kaggle/working && rm -f cuc_runs.zip && zip -qr cuc_runs.zip cuc_runs && ls -lh cuc_runs.zip